In [18]:
from typing import Dict
import json

import numpy as np
import torch
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
import torchvision.transforms.v2 as tv_v2

from flower.models.flower import FLOWERVLA

In [19]:
episode_data_calvin = np.load(r"C:\Users\mohes\Documents\Coding\flower_vla_calvin\data\calvin_debug_dataset\calvin_debug_dataset\validation\episode_0553567.npz")
lang_ann_calvin = np.load(r"C:\Users\mohes\Documents\Coding\flower_vla_calvin\data\calvin_debug_dataset\calvin_debug_dataset\validation\lang_annotations\auto_lang_ann.npy", allow_pickle=True)

episode_data_calvin = dict(episode_data_calvin)
lang_ann_calvin = dict(lang_ann_calvin[()])

In [20]:
def pprint_nested_dict_structure(dict_to_analyze: Dict) -> str:
    """Gets a string representing the structure of a nested dictionary.
    If the leaf value is a numpy array, the shape is printed to make the output more
    informative.

    Args:
        dict_to_analyze (Dict): The dictionary to analyze.

    Returns:
        str: The string representing the structure of the dictionary.
    """

    def _pprint_nested_dict_structure(d, level=0):
        result = ""
        for key, value in d.items():
            if isinstance(value, dict):
                result += f"{'  ' * level}{key}:\n"
                result += _pprint_nested_dict_structure(value, level + 1)

            else:  # habdle numpy arrays and torch tensors
                try:
                    result += f"{'  ' * level}{key}: {type(value)} {value.shape} {value.dtype}"
                except AttributeError:
                    result += f"{'  ' * level}{key}: {value}"
                try:
                    result += f" {value.device}"
                except AttributeError:
                    pass
                result += "\n"

        return result

    return _pprint_nested_dict_structure(dict_to_analyze)

In [21]:
print(pprint_nested_dict_structure(lang_ann_calvin))
print(pprint_nested_dict_structure(episode_data_calvin))

language:
  ann: ['lift the red block from the table', 'turn on the light bulb', 'in the slider pick up the blue block', 'in the cabinet grasp the blue block', 'slide down the switch', 'put it in the slider', 'slide right the pink block', 'in the slider grasp the blue block']
  task: ['lift_red_block_table', 'turn_on_lightbulb', 'lift_blue_block_slider', 'lift_blue_block_slider', 'turn_off_lightbulb', 'place_in_slider', 'push_pink_block_right', 'lift_blue_block_slider']
  emb: <class 'numpy.ndarray'> (8, 1, 384) float32
info:
  episodes: []
  indx: [(554046, 554110), (553636, 553700), (553693, 553757), (553691, 553755), (554474, 554538), (554110, 554145), (553916, 553961), (553701, 553765)]

actions: <class 'numpy.ndarray'> (7,) float64
rel_actions: <class 'numpy.ndarray'> (7,) float64
robot_obs: <class 'numpy.ndarray'> (15,) float64
scene_obs: <class 'numpy.ndarray'> (24,) float64
rgb_static: <class 'numpy.ndarray'> (200, 200, 3) uint8
rgb_gripper: <class 'numpy.ndarray'> (84, 84, 3) 

In [22]:
flower_model_basic = FLOWERVLA()

Loading Florence-2 from microsoft/Florence-2-base


Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))


In [ ]:
repo_id = "mbreuss/flower_calvin_abcd"

print("Downloading config...")
config_path = hf_hub_download(repo_id=repo_id, filename="config.json")
with open(config_path, "r") as f:
    config = json.load(f)

print("Initializing model...")
flower_model_hf_abcd = FLOWERVLA(**config["model_config"])

print("Downloading weights...")
weights_path = hf_hub_download(repo_id=repo_id, filename="model.safetensors")
state_dict = load_file(weights_path)

# rename all keys in the state dict that have the form vlm.language_encoder.something to vlm.language_model.model.encoder.something
state_dict_renamed = {
    k.replace("vlm.language_encoder.", "vlm.language_model.model.encoder."): v
    for k, v in state_dict.items()
}
state_dict_renamed["vlm.language_model.final_logits_bias"] = state_dict_renamed.pop(
    "vlm.language_final_logits_bias"
)
state_dict_renamed["vlm.language_model.model.shared.weight"] = state_dict_renamed.pop(
    "vlm.language_shared.weight"
)

print("Loading weights...")
flower_model_hf_abcd.load_state_dict(state_dict_renamed)

In [ ]:
repo_id = "mbreuss/flower_vla_pret"

print("Downloading weights...")
weights_path = hf_hub_download(repo_id=repo_id, filename="360000_model_weights.pt")
print(weights_path)
print("Initializing model...")
flower_model_hf_pret = FLOWERVLA(
    load_pretrained=True,
    pretrained_model_path=weights_path,
    vlm_path="microsoft/Florence-2-large",
    token_dropout=0.1,
    num_sampling_steps=4,
    use_second_view=True,
    sampling_type="uniform",
    dit_dim=1024,
    n_layers=18,
    use_rope=True,
    query_seq_len=100,
)

In [23]:
obs = {
    "rgb_obs": {
        # B, T, C, H, W
        "rgb_static": tv_v2.Resize((224, 224))(
            torch.from_numpy(episode_data_calvin["rgb_static"]).permute(2, 1, 0)[
                None, None
            ]
        ),
        "rgb_gripper": tv_v2.Resize((224, 224))(
            torch.from_numpy(episode_data_calvin["rgb_gripper"]).permute(2, 1, 0)[
                None, None
            ]
        ),
    }
}

In [ ]:
flower_model_basic.step(obs, goal = {"lang_text": "pick up the blue cube"})

In [ ]:
flower_model.obs_modalities
flower_model.use_proprio

In [ ]:
# {"rgb_obs": {"rgb_gripper": , "rgb_static"}, 
#  "lang_text": }